# Hansen《Econometrics》第 3 章习题解答

**Chapter 3 The Algebra of Least Squares**

对应书稿 PDF 第 112–116 页（印刷页 92–96），§3.26 Exercises。

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch03_Exercises_Solutions.md`（强烈建议先读它的 §0、§1 概念铺垫）。本 notebook：理论摘要 + **Exercise 3.24–3.26 数值计算**。

> **写给只学过李子奈/陈强的同学：** 本章把 OLS **全部**写成两个矩阵 $P=X(X'X)^{-1}X'$（投影/hat 矩阵）和 $M=I-P$（零化矩阵）的语言。记两句口诀，本章 90% 的题迎刃而解：
> 1. **"$P$ 保留 col($X$)，$M$ 消灭 col($X$)"**：$PX=X$、$MX=0$。
> 2. **"$P$、$M$ 幂等"**：$P^2=P$、$M^2=M$、$PM=0$。
>
> 配合 $\hat Y=PY$、$\hat e=MY$、$X'\hat e=0$（正规方程），所有 OLS 性质统一成"正交"这一个概念。本章**没有概率/统计**（那是第 4 章起），纯粹是线性代数。


## Exercise 3.1–3.11 代数核心（一句话直觉 + 结论）

| 题 | 结论 | 直觉（$P$/$M$ 视角） |
|:--:|------|------|
| 3.1 | 矩条件 $\Rightarrow$ 样本均值与 $1/n$ 方差 | 矩方法：两方程解两未知数（Ch2 Ex 2.17 的样本版） |
| 3.2 | $Z=XC$ 可逆 $\Rightarrow$ 拟合、残差不变，$\hat\beta_Z=C^{-1}\hat\beta_X$ | col($Z$)=col($X$)，投影空间不变 |
| 3.3 | $X'\hat e=0$ | 正规方程；因 $X'M=0$ |
| 3.4 | $X_2'\hat e=0$ | $X'\hat e=0$ 的分块 |
| 3.5 | $\hat e$ 对 $X$ 回归系数 $=0$ | 残差已"榨干"$X$ 的线性信息 |
| 3.6 | $\hat Y$ 对 $X$ 回归系数 $=\hat\beta$ | $\hat Y\in$col($X$)，再投影不变 |
| 3.7 | $PX_1=X_1$，$MX_1=0$ | "$P$ 保留、$M$ 消灭" col($X$) |
| 3.8 | $M^2=M$ | $M=I-P$，用 $P^2=P$ |
| 3.9 | $\mathrm{tr}(M)=n-k$ | 自由度 $n-k$ 的代数来源 |
| 3.10 | $X_1'X_2=0\Rightarrow P=P_1+P_2$ | 正交分块→投影可拆（多重共线性的反面） |
| 3.11 | 含截距 $\Rightarrow\overline{\hat Y}=\bar Y$ | 截距→$\sum\hat e_i=0$→残差均值为 0 |

## Exercise 3.12–3.23（详见 .md）

- **3.12** 虚拟变量陷阱：$D_1+D_2=\iota$ 与截距共线；(3.52) 不可估，(3.53)(3.54) 列空间相同、拟合一致。
- **3.13** 组内去均值 + FWL = 固定效应/within 估计量雏形（Ch.17 基础）；$\tilde\beta=\hat\beta$。
- **3.14** Sherman–Morrison 在线更新：$\hat\beta_{n+1}=\hat\beta_n+\text{修正}\times(Y_{n+1}-X_{n+1}'\hat\beta_n)$。
- **3.15** 含截距时 $R^2=\mathrm{Corr}(Y,\hat Y)^2$。
- **3.16** 嵌套回归 $R_2^2\ge R_1^2$；相等 iff 新变量系数为 0（→调整 $R^2$ 的动机）。
- **3.17** $\tilde\sigma^2\ge\hat\sigma^2$（留一误差 ≥ 样本内残差，高杠杆点放大更多）。
- **3.18** $\hat\beta_{(-i)}=\hat\beta$ iff $\hat e_i=0$（影响点判据）。
- **3.19** 截距模型：$h_{ii}=1/n$，$\tilde e_i=\frac{n}{n-1}(Y_i-\bar Y)$。
- **3.20** $\hat\sigma^2_{(-i)}=\frac{n}{n-1}\hat\sigma^2-\frac{\hat e_i^2}{(n-1)(1-h_{ii})}$（推导见 .md，用幂等性）。
- **3.21** 正交回归元（$X_1'X_2=0$）→ 可逐个回归，系数与联合回归相同。
- **3.22** 只把 $Y$ 对 $X_1$ 净化、再回归于 $X_2$ **是错的**（FWL 要求 $Y$ 和 $X_2$ 都净化）。
- **3.23** $Z=[X_1,X_2-X_1]=XC$ 可逆 → 残差相同 → $\hat\sigma^2=\tilde\sigma^2$。


## 准备数据：方程 (3.49) 样本

单身亚裔男性（`race==4`, `marital==7`, `female==0`），`experience < 45`。


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

candidates = [
    Path("../../hansen/econometrics/data/cps09mar/cps09mar.xlsx"),
    Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx"),
]
DATA = next(p for p in candidates if p.exists())
print("data:", DATA.resolve())

df = pd.read_excel(DATA)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"] / (df["hours"] * df["week"]))
df["exp2"] = (df["experience"] ** 2) / 100

mask_349 = (
    (df["race"] == 4)
    & (df["marital"] == 7)
    & (df["female"] == 0)
    & (df["experience"] < 45)
)
sub = df.loc[mask_349].copy()
print("n =", len(sub))
sub[["education", "experience", "lwage"]].describe()


## Exercise 3.24 (a) 估计 (3.49)


In [ ]:
def ols(y, X, names=None):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    yhat = X @ beta
    sse = float(np.sum(e ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - sse / sst
    idx = names if names is not None else list(range(len(beta)))
    return pd.Series(beta, index=idx), e, yhat, r2, sse

y = sub["lwage"].to_numpy(float)
X = np.column_stack([
    sub["education"].to_numpy(float),
    sub["experience"].to_numpy(float),
    sub["exp2"].to_numpy(float),
    np.ones(len(sub)),
])
names = ["education", "experience", "exp2/100", "intercept"]
beta, e, yhat, r2, sse = ols(y, X, names)
print(beta.to_string())
print(f"R^2 = {r2:.6f}")
print(f"SSE = {sse:.6f}")
print("Book (3.49) approx: 0.144, 0.043, -0.095, 0.531")


## Exercise 3.24 (b)(c) FWL 残差回归

**FWL 定理（核心工具）：** 分块 $X=[X_1\ X_2]$，$\hat\beta_2$ 等于"把 $Y$ 和 $X_2$ **都**对 $X_1$ 取残差后，残差对残差回归"的系数。

**本题的 $X_1$ = (experience, exp², 截距)，$X_2$ = education。** 三步：
1. log(wage) 对 $X_1$ 回归 → 残差 $\tilde Y$（"剥掉经验影响的工资"）；
2. education 对 $X_1$ 回归 → 残差 $\tilde X_1$（"剥掉经验影响的教育"）；
3. $\tilde Y$ 对 $\tilde X_1$ 回归 → 斜率应 = (a) 中的 0.1443，SSE 也 = (a) 的 82.505。

**注意（Ex 3.22 的教训）：** 必须 $Y$ **和** $X_2$ 都净化；只净化 $Y$ 是错的。下方代码两步都做。


In [ ]:
Z = np.column_stack([
    sub["experience"].to_numpy(float),
    sub["exp2"].to_numpy(float),
    np.ones(len(sub)),
])
edu = sub["education"].to_numpy(float)

b_y = np.linalg.lstsq(Z, y, rcond=None)[0]
b_e = np.linalg.lstsq(Z, edu, rcond=None)[0]
ry = y - Z @ b_y
re = edu - Z @ b_e

Xfwl = np.column_stack([re, np.ones(len(re))])
beta_fwl, e_fwl, _, r2_fwl, sse_fwl = ols(ry, Xfwl, ["education (FWL)", "intercept"])
print(beta_fwl.to_string())
print(f"FWL R^2 = {r2_fwl:.6f}")
print(f"FWL SSE = {sse_fwl:.6f}")
print("slope equals full OLS?", np.isclose(beta_fwl.iloc[0], beta["education"]))
print("SSE equals full OLS?", np.isclose(sse_fwl, sse))
print("R^2 equal? (expect False)", np.isclose(r2_fwl, r2))


## Exercise 3.25 数值核对 OLS 性质


In [ ]:
X1 = sub["education"].to_numpy(float)
X2 = sub["experience"].to_numpy(float)

checks = {
    "(a) sum e": np.sum(e),
    "(b) sum X1*e": np.sum(X1 * e),
    "(c) sum X2*e": np.sum(X2 * e),
    "(d) sum X1^2*e": np.sum(X1**2 * e),
    "(e) sum X2^2*e": np.sum(X2**2 * e),
    "(f) sum Yhat*e": np.sum(yhat * e),
    "(g) sum e^2": np.sum(e**2),
}
for k, v in checks.items():
    print(f"{k:20s} = {v: .6e}")

print()
print("Theory notes:")
print("- (a)(b)(c)(f) ~ 0: intercept, edu, exp in X; Yhat in col(X)")
print("- (e) ~ 0: exp^2/100 is in the regression")
print("- (d) != 0: education^2 not in the regression")
print("- (g) = SSE")


## Exercise 3.26 白人男性西班牙裔 log wage 回归

基准：Midwest；婚姻基准：never married；widowed 与 divorced 合并。


In [ ]:
mask_w = (df["race"] == 1) & (df["female"] == 0) & (df["hisp"] == 1)
s2 = df.loc[mask_w].copy()
print("n white male Hispanic =", len(s2))

s2["married"] = s2["marital"].isin([1, 2, 3]).astype(float)
s2["wid_div"] = s2["marital"].isin([4, 5]).astype(float)
s2["separated"] = (s2["marital"] == 6).astype(float)
s2["NE"] = (s2["region"] == 1).astype(float)
s2["South"] = (s2["region"] == 3).astype(float)
s2["West"] = (s2["region"] == 4).astype(float)

y2 = s2["lwage"].to_numpy(float)
X2m = np.column_stack([
    s2["education"], s2["experience"], s2["exp2"],
    s2["NE"], s2["South"], s2["West"],
    s2["married"], s2["wid_div"], s2["separated"],
    np.ones(len(s2)),
])
names2 = [
    "education", "experience", "exp2/100",
    "Northeast", "South", "West",
    "married", "widowed/divorced", "separated",
    "intercept",
]
beta2, e2, yhat2, r2_2, sse2 = ols(y2, X2m, names2)
print(beta2.to_string())
print(f"R^2 = {r2_2:.6f}, SSE = {sse2:.6f}")


### 3.26 (b) 与 statsmodels 对照


In [ ]:
try:
    import statsmodels.api as sm
    res = sm.OLS(y2, X2m).fit()
    cmp = pd.DataFrame({
        "numpy_lstsq": beta2.values,
        "statsmodels": res.params,
        "abs_diff": np.abs(beta2.values - res.params),
    }, index=names2)
    display(cmp) if "display" in dir() else print(cmp)
    print("max abs diff =", float(cmp["abs_diff"].max()))
except Exception as ex:
    print("statsmodels unavailable or error:", ex)
    print(beta2)


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格用模拟核对 ch03 的关键结论：投影/幂等矩阵、$R^2$、留一（LOO）残差与方差公式。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(3)

# 基本设定：含截距的设计矩阵 X，拟合后得到投影 P、零化 M、杠杆 h、残差 e
n, k = 80, 4
X = np.column_stack([rng.standard_normal((n, k-1)), np.ones(n)])
beta = np.array([1.0, -0.5, 2.0, 0.3])
Y = X @ beta + rng.standard_normal(n) * 2.0
XtXinv = np.linalg.inv(X.T @ X)
P = X @ XtXinv @ X.T
M = np.eye(n) - P
beta_hat = np.linalg.solve(X.T @ X, X.T @ Y)
e = Y - X @ beta_hat
h = np.diag(P)
print("[基本] X'e≈0:", np.allclose(X.T @ e, 0),
      " P²=P:", np.allclose(P @ P, P),
      " tr(P)=k:", np.isclose(np.trace(P), k),
      " tr(M)=n-k:", np.isclose(np.trace(M), n-k))

# Ex 3.15: 含截距时 R² = Corr(Y, Ŷ)²
yhat = X @ beta_hat
r2 = 1 - (e @ e) / ((Y - Y.mean()) @ Y)
print(f"[3.15] R²={r2:.4f}  Corr²={np.corrcoef(Y, yhat)[0,1]**2:.4f}  (应相等)")

# Ex 3.17: 留一误差 σ̃² ≥ 样本内残差 σ̂²（高杠杆点被放大）
sig2hat = (e @ e) / n
sig2tilde = np.mean((e / (1 - h))**2)
print(f"[3.17] σ̂²={sig2hat:.4f} ≤ σ̃²={sig2tilde:.4f}: {sig2tilde >= sig2hat}")

# Ex 3.18 & 3.20: 删一估计（暴力删行 vs 公式）
i = 3
Xmi, Ymi = np.delete(X, i, 0), np.delete(Y, i)
beta_loo = np.linalg.solve(Xmi.T @ Xmi, Xmi.T @ Ymi)
e_loo_pred = e[i] / (1 - h[i])                                  # Theorem 3.7 的预测误差
sse_loo = ((Ymi - Xmi @ beta_loo)**2).sum()
sig2loo_formula = n / (n-1) * sig2hat - e[i]**2 / ((n-1) * (1 - h[i]))   # Ex 3.20 公式
print(f"[3.18] β̂(-i) 与 β̂ 的差={np.abs(beta_loo - beta_hat).max():.2e} (ê_i={e[i]:.3f}; 差=0 iff ê_i=0)")
print(f"[3.20] σ²(-i) 暴力={sse_loo/(n-1):.5f}  公式={sig2loo_formula:.5f}  (应一致)")

# Ex 3.19: 截距模型 LOO = n/(n-1)(Y - Ȳ)
Yc = rng.standard_normal(n) * 2 + 5
eh = Yc - Yc.mean()
print(f"[3.19] 截距LOO匹配: {np.allclose(eh / (1 - 1/n), n/(n-1) * (Yc - Yc.mean()))}")

# Ex 3.10: 回归元正交分块时 P = P1 + P2
X1 = rng.standard_normal((n, 2))
Z = rng.standard_normal((n, 2))
X2 = Z - X1 @ np.linalg.solve(X1.T @ X1, X1.T @ Z)              # 残差化使 X1'X2 = 0
Xo = np.column_stack([X1, X2])
Po = Xo @ np.linalg.inv(Xo.T @ Xo) @ Xo.T
P1 = X1 @ np.linalg.solve(X1.T @ X1, X1.T)
P2 = X2 @ np.linalg.solve(X2.T @ X2, X2.T)
print(f"[3.10] X1'X2≈0: {np.allclose(X1.T @ X2, 0)}   P=P1+P2: {np.allclose(Po, P1 + P2)}")